In [ ]:
# TV OFF / ACE-Step 1.5 — Original Music Library Track 003 (non-publishing)
# Kaggle's current base image is Python 3.13; upstream ACE-Step currently
# requires Python <3.13. We therefore create an isolated Python 3.12 venv.
import os, sys, json, subprocess, shutil
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = Path("/tmp/ACE-Step-1.5")
OUT = WORK / "ace_step_output"
OUT.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None):
    print("+", " ".join(map(str, cmd)), flush=True)
    child_env = os.environ.copy()
    # Kaggle's Jupyter process exports an inline Matplotlib backend that is
    # unavailable in the isolated Python 3.12 venv. Use a headless backend.
    child_env["MPLBACKEND"] = "Agg"
    subprocess.run(list(map(str, cmd)), cwd=str(cwd) if cwd else None, env=child_env, check=True)

# Confirm that Kaggle has the expected CUDA hardware before installing packages.
import torch
print("Kaggle Python:", sys.version)
print("Kaggle PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable; refusing CPU fallback.")
gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
for i, name in enumerate(gpu_names):
    print("GPU", i, name, round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 2), "GiB")
if not any("T4" in name for name in gpu_names):
    raise RuntimeError(f"Expected Tesla T4; detected {gpu_names}")

# Pin to the actual upstream latest release tag (v0.1.8), not a nonexistent v1.5.2 tag.
if not REPO.exists():
    run(["git", "clone", "--depth", "1", "--branch", "v0.1.8",
         "https://github.com/ace-step/ACE-Step-1.5.git", REPO])

# The upstream v0.1.8 release hardcodes float16 on pre-Ampere CUDA GPUs.
# Its NaN error hint mentions ACESTEP_DTYPE, but this release does not read
# that variable. Apply a guarded local patch to enable float32 for the T4.
orchestrator = REPO / "acestep/core/generation/handler/init_service_orchestrator.py"
orchestrator_text = orchestrator.read_text(encoding="utf-8")
old_dtype_block = '''            elif resolved_device == "cuda":
                if gpu_config.cuda_supports_bfloat16():
                    self.dtype = torch.bfloat16
                else:
                    self.dtype = torch.float16
                    logger.info(
                        "[initialize_service] Pre-Ampere CUDA detected: "
                        "using float16 instead of bfloat16."
                    )'''
new_dtype_block = '''            elif resolved_device == "cuda":
                forced_dtype = os.environ.get("ACESTEP_DTYPE", "").strip().lower()
                if forced_dtype == "float32":
                    self.dtype = torch.float32
                    logger.info("[initialize_service] ACESTEP_DTYPE=float32 override applied.")
                elif gpu_config.cuda_supports_bfloat16():
                    self.dtype = torch.bfloat16
                else:
                    self.dtype = torch.float16
                    logger.info(
                        "[initialize_service] Pre-Ampere CUDA detected: "
                        "using float16 instead of bfloat16."
                    )'''
if old_dtype_block not in orchestrator_text:
    raise RuntimeError("Expected v0.1.8 CUDA dtype block not found; refusing an unsafe source patch.")
orchestrator.write_text(orchestrator_text.replace(old_dtype_block, new_dtype_block, 1), encoding="utf-8")
os.environ["ACESTEP_DTYPE"] = "float32"
print("Applied guarded ACE-Step T4 float32 override:", orchestrator)

# ACE-Step declares Python >=3.11,<3.13. Kaggle's host is Python 3.13,
# so use an isolated 3.12 runtime instead of patching upstream metadata.
uv = Path(shutil.which("uv") or "/usr/local/bin/uv")
if not uv.exists():
    run(["bash", "-lc", "curl -LsSf https://astral.sh/uv/install.sh | sh"])
if not uv.exists():
    raise RuntimeError(f"uv executable was not found; checked PATH and /usr/local/bin. PATH={os.environ.get('PATH')}")
run([uv, "python", "install", "3.12"])
venv = REPO / ".venv"
run([uv, "venv", "--python", "3.12", str(venv)], cwd=REPO)
python = venv / "bin" / "python"
run([uv, "pip", "install", "--python", str(python), "-e", "."], cwd=REPO)

# Generate using the isolated supported Python runtime.
generation_script = r'''
import json, math, shutil, hashlib
from pathlib import Path
import numpy as np
import soundfile as sf
import torch
from acestep.handler import AceStepHandler
from acestep.inference import GenerationParams, GenerationConfig, generate_music

OUT = Path("/kaggle/working/ace_step_output")
dit = AceStepHandler()
message, initialized = dit.initialize_service(
    project_root="/tmp/ACE-Step-1.5",
    config_path="acestep-v15-turbo",
    device="cuda",
    offload_to_cpu=True,
    quantization="int8_weight_only",
)
print("ACE-Step initialization:", message, flush=True)
if not initialized:
    raise RuntimeError("ACE-Step DiT initialization failed.")

params = GenerationParams(
    task_type="text2music",
    caption=(
        "Original instrumental contemporary bluegrass string-band composition in D major, bright and spirited, "
        "a nimble clawhammer banjo carrying a memorable syncopated pentatonic melody, answered by lively twin fiddle harmonies, "
        "resonant upright acoustic bass playing a driving two-beat boom-chuck pulse, flatpicked steel-string acoustic guitar providing rhythmic chop, mandolin offbeat chops and rolling banjo fills, "
        "a clear AABBCCBA form with a four-bar pickup, a fiddle-led B variation, a banjo-and-mandolin breakdown, a lively fiddle solo, and a strong final A-theme return, I-IV-V harmony with a brief vi turn, melodic call-and-response, tight acoustic ensemble dynamics, "
        "professional studio production, no vocals, no speech, no sound effects"
    ),
    lyrics="",  # Empty lyrics are required for stable instrumental generation on T4.
    instrumental=True,
    vocal_language="unknown",
    bpm=132,
    duration=180,
    inference_steps=8,
    shift=3.0,
    seed=20261123,
    thinking=False,
    use_cot_metas=False,
    use_cot_caption=False,
    use_cot_language=False,
)
config = GenerationConfig(
    batch_size=1,
    allow_lm_batch=False,
    use_random_seed=False,
    seeds=[20261123],
    audio_format="wav",
)
result = generate_music(dit, None, params, config, save_dir=str(OUT))
if not result.success:
    raise RuntimeError(f"ACE-Step generation failed: {result.error}")
if not result.audios:
    raise RuntimeError("ACE-Step returned success but no audio output.")

source = Path(result.audios[0]["path"])
target = OUT / "TV_OFF_Original_Music_003_Copperline_Run_Bluegrass.wav"
if not source.is_file():
    raise RuntimeError(f"Audio file not found: {source}")
if source.resolve() != target.resolve():
    shutil.copy2(source, target)

info = sf.info(str(target))
audio, sr = sf.read(str(target), always_2d=True, dtype="float32")
peak = float(np.max(np.abs(audio)))
rms = float(np.sqrt(np.mean(np.square(audio))))
duration = float(info.duration)
if not 179.0 <= duration <= 181.0:
    raise RuntimeError(f"Unexpected audio duration: {duration:.2f}s")
if sr != 48000 or info.channels != 2 or info.subtype != "PCM_16":
    raise RuntimeError(f"Unexpected format: {sr} Hz, {info.channels} channels, {info.subtype}")
if not np.isfinite(audio).all():
    raise RuntimeError("Audio contains NaN or infinite samples.")
if peak < 0.01 or peak > 1.001:
    raise RuntimeError(f"Invalid peak level: {peak:.5f}")
if peak >= 1.0:
    raise RuntimeError(f"Clipping detected: peak={peak:.6f}")
if rms < 0.01 or rms > 0.35:
    raise RuntimeError(f"Audio RMS too low: {rms:.6f}")

manifest = {
    "status": "ok",
    "provider": "ACE-Step-1.5",
    "release": "v0.1.8",
    "model": "acestep-v15-turbo",
    "python": __import__("sys").version,
    "gpu": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "cuda": True,
    "duration_seconds": round(duration, 3),
    "sample_rate": int(sr),
    "channels": int(info.channels),
    "subtype": info.subtype,
    "format": "WAV PCM_16",
    "track_number": 3,
    "peak": round(peak, 6),
    "rms": round(rms, 6),
    "seed": 20261123,
    "title": "Copperline Run — Bluegrass String Band", 
    "style": "Contemporary bluegrass / acoustic string-band instrumental", 
    "prompt": params.caption,
    "bpm": 132,
    "inference_steps": 8,
    "instrumental": True,
    "publishing_enabled": False,
    "audio_file": target.name,
    "sha256": hashlib.sha256(target.read_bytes()).hexdigest(),
}
# Create an MP3 convenience copy if ffmpeg is available.
import shutil as _shutil, subprocess as _subprocess
ffmpeg = _shutil.which("ffmpeg")
if ffmpeg:
    mp3 = OUT / "TV_OFF_Original_Music_003_Copperline_Run_Bluegrass.mp3"
    _subprocess.run([ffmpeg, "-y", "-i", str(target), "-codec:a", "libmp3lame", "-b:a", "320k", str(mp3)], check=True, stdout=_subprocess.DEVNULL, stderr=_subprocess.PIPE)
    manifest["mp3_file"] = mp3.name
else:
    print("ffmpeg unavailable; WAV master retained, MP3 copy skipped.")
(OUT / "ace_step_smoke_result.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
print("TEST PASSED — audio generated and validated; nothing was published.")
'''
script_path = WORK / "run_ace_step_smoke.py"
script_path.write_text(generation_script, encoding="utf-8")
run([python, str(script_path)])
print("ACE-Step test process finished.")
